# HW1 - Seeded Circuits, Bit Flips, and Measurement Mapping

**Course context:** AI-resilient Qiskit homework prototype.

In this assignment, your output depends on your student-specific seed. You will build a small circuit, apply a seeded bit-flip layer, measure using a seeded measurement map, and export `answers.json`.

The goal is not just to get a histogram. The goal is to prove you understand how the displayed Qiskit bitstring relates to qubits and classical bits.

## Learning goals

By the end, you should be able to:

1. Generate a personalized Qiskit circuit from a deterministic seed.
2. Prepare a computational basis state using X gates.
3. Apply a deterministic bit-flip mask.
4. Measure qubits into classical bits using a nontrivial mapping.
5. Explain why Qiskit count strings are displayed as `c[n-1]...c[0]`.
6. Export machine-readable `answers.json` for an autograder.

## AI-resilience idea

A text-only LLM may write generic Qiskit code, but it may fail if it does not actually run your seeded circuit or if it confuses bit order. Your personalized seed controls the bitstring, bit flips, and measurement mapping.

In [ ]:
# Setup. In Colab, run this cell first.
%pip -q install qiskit qiskit-aer matplotlib

## 1. Enter your student ID

Use your real student identifier if assigned by the instructor. For testing, any string works. Do not change it after generating your config, because it changes the expected answer.

In [ ]:
STUDENT_ID = "replace_with_your_student_id"
ASSIGNMENT_ID = "HW1"
SHOTS = 2048

## 2. Generate your personalized configuration

Conventions used in this assignment:

- `initial_bits_q0_to_qn` is listed in qubit-index order: `q[0], q[1], ..., q[n-1]`.
- `flip_mask_q0_to_qn` tells you which qubits receive an additional X gate.
- `final_bits_q0_to_qn = initial_bits XOR flip_mask`.
- `measurement_map` is a list of `[qubit_index, classical_bit_index]` pairs.
- Qiskit displays count keys as `c[n-1]...c[0]`.

This means the displayed count string may not look like the qubit order you expect, especially if the measurement map is reversed.

In [ ]:
import hashlib
import json
from pathlib import Path


def stable_int_seed(student_id: str, assignment_id: str = "HW1") -> int:
    raw = f"{assignment_id}::{student_id}".encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:16], 16)


def _lcg(seed: int):
    state = seed % (2**31 - 1)
    while True:
        state = (1103515245 * state + 12345) % (2**31 - 1)
        yield state


def _non_palindromic_bits(gen, n: int):
    while True:
        bits = [next(gen) % 2 for _ in range(n)]
        if any(bits) and bits != list(reversed(bits)):
            return bits


def generate_config(student_id: str, assignment_id: str = "HW1"):
    seed = stable_int_seed(student_id, assignment_id)
    gen = _lcg(seed)
    n = 4 + (next(gen) % 2)
    initial_bits = _non_palindromic_bits(gen, n)
    flip_mask = [0] * n
    first_flip = next(gen) % n
    flip_mask[first_flip] = 1
    if next(gen) % 3 == 0:
        second_flip = next(gen) % n
        flip_mask[second_flip] ^= 1
    final_bits = [a ^ b for a, b in zip(initial_bits, flip_mask)]
    if final_bits == list(reversed(final_bits)):
        flip_mask[0] ^= 1
        final_bits = [a ^ b for a, b in zip(initial_bits, flip_mask)]
    measurement_mode = "direct" if (next(gen) % 2 == 0) else "reversed"
    if measurement_mode == "direct":
        measurement_map = [[i, i] for i in range(n)]
    else:
        measurement_map = [[i, n - 1 - i] for i in range(n)]
    return {
        "assignment_id": assignment_id,
        "student_id": student_id,
        "seed": seed,
        "num_qubits": n,
        "initial_bits_q0_to_qn": initial_bits,
        "flip_mask_q0_to_qn": flip_mask,
        "final_bits_q0_to_qn": final_bits,
        "measurement_mode": measurement_mode,
        "measurement_map": measurement_map,
        "bit_order_note": "Counts are displayed as c[n-1]...c[0], not q[0]...q[n-1].",
    }

config = generate_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

## 3. Build your circuit

Create a circuit with `n` qubits and `n` classical bits.

Your circuit must:

1. Apply X gates according to `initial_bits_q0_to_qn`.
2. Apply a barrier.
3. Apply X gates according to `flip_mask_q0_to_qn`.
4. Apply a barrier.
5. Measure using `measurement_map`.

Do **not** use `measure_all()` for this assignment because it may hide the custom measurement map.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

n = config["num_qubits"]
q = QuantumRegister(n, "q")
c = ClassicalRegister(n, "c")
qc = QuantumCircuit(q, c)

# TODO 1: Prepare the initial basis state.
# If initial_bits_q0_to_qn[i] == 1, apply X to q[i].
for i, bit in enumerate(config["initial_bits_q0_to_qn"]):
    # replace pass with your code
    pass

qc.barrier(label="seeded initial state")

# TODO 2: Apply the seeded bit-flip mask.
# If flip_mask_q0_to_qn[i] == 1, apply X to q[i].
for i, bit in enumerate(config["flip_mask_q0_to_qn"]):
    # replace pass with your code
    pass

qc.barrier(label="seeded bit flips")

# TODO 3: Measure according to measurement_map.
# Each pair is [qubit_index, classical_bit_index].
for q_index, c_index in config["measurement_map"]:
    # replace pass with your code
    pass

qc.draw("mpl")

## 4. Compute the expected displayed bitstring

You should compute this yourself rather than hard-coding the answer.

Remember: Qiskit displays count strings as `c[n-1]...c[0]`.

In [ ]:
def expected_display_bitstring(config):
    n = config["num_qubits"]
    final_bits = config["final_bits_q0_to_qn"]
    classical_bits = [0] * n
    for q_index, c_index in config["measurement_map"]:
        classical_bits[c_index] = final_bits[q_index]
    return "".join(str(classical_bits[i]) for i in reversed(range(n)))


def expected_logical_q_string(config):
    final_bits = config["final_bits_q0_to_qn"]
    return "".join(str(final_bits[i]) for i in reversed(range(len(final_bits))))

expected_display = expected_display_bitstring(config)
expected_logical = expected_logical_q_string(config)
print("Expected logical qubit string q[n-1]...q[0]:", expected_logical)
print("Expected displayed count string c[n-1]...c[0]:", expected_display)

## 5. Run the simulator

Your simulator output should have the expected displayed bitstring as the dominant result. Since this is a basis-state circuit without superposition, it should appear in essentially all shots.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram

sim = AerSimulator(seed_simulator=config["seed"] % (2**32 - 1))
tqc = transpile(qc, sim, seed_transpiler=config["seed"] % (2**32 - 1))
result = sim.run(tqc, shots=SHOTS).result()
counts = result.get_counts()

dominant_bitstring = max(counts, key=counts.get)
print("Counts:", counts)
print("Dominant bitstring:", dominant_bitstring)
print("Expected displayed bitstring:", expected_display)
plot_histogram(counts)

## 6. Reflection questions

Answer both questions in complete sentences.

1. **Bit order:** Explain how your `measurement_map` determines the displayed Qiskit count string.
2. **Bit flips:** Explain how the seeded bit-flip mask changed the initial bit pattern into the final bit pattern.

In [ ]:
reflection_bit_order = "TODO: Explain how q[i] maps to c[j], and why counts are displayed as c[n-1]...c[0]."
reflection_bit_flip = "TODO: Explain how initial_bits XOR flip_mask produced final_bits."

## 7. Export `answers.json`

Submit this file. Your instructor may also request the notebook and/or circuit image.

In [ ]:
answers = {
    "assignment_id": ASSIGNMENT_ID,
    "student_id": STUDENT_ID,
    "seed": config["seed"],
    "num_qubits": config["num_qubits"],
    "shots": SHOTS,
    "measurement_mode": config["measurement_mode"],
    "measurement_map": config["measurement_map"],
    "initial_bits_q0_to_qn": config["initial_bits_q0_to_qn"],
    "flip_mask_q0_to_qn": config["flip_mask_q0_to_qn"],
    "final_bits_q0_to_qn": config["final_bits_q0_to_qn"],
    "expected_logical_qn_to_q0": expected_logical,
    "expected_display_bitstring": expected_display,
    "counts": {k: int(v) for k, v in counts.items()},
    "dominant_bitstring": dominant_bitstring,
    "reflection_bit_order": reflection_bit_order,
    "reflection_bit_flip": reflection_bit_flip,
}

with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)

print(json.dumps(answers, indent=2))
print("Saved answers.json")

## Optional: Export circuit as QPY

Your instructor may use QPY to validate the circuit object directly instead of relying only on screenshots or JSON.

In [ ]:
# Optional QPY export.
# from qiskit import qpy
# with open("circuit.qpy", "wb") as f:
#     qpy.dump(qc, f)
# print("Saved circuit.qpy")